In [26]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, concatenate, Add, Flatten, Dense, BatchNormalization, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.manifold import TSNE
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.optimizers import Adam

In [27]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0
y_train, y_test = to_categorical(y_train, 10), to_categorical(y_test, 10)

In [ ]:
def inception_module(x, filters):
    conv1x1 = Conv2D(filters[0], (1, 1), padding='same', activation='relu')(x)
    conv3x3 = Conv2D(filters[1], (3, 3), padding='same', activation='relu')(x)
    conv5x5 = Conv2D(filters[2], (5, 5), padding='same', activation='relu')(x)
    maxpool = MaxPooling2D((3, 3), strides=(1, 1), padding='same')(x)
    return concatenate([conv1x1, conv3x3, conv5x5, maxpool], axis=-1)

def residual_block(x, filters):
    shortcut = x
    x = Conv2D(filters, (3, 3), padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = Conv2D(filters, (3, 3), padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    return x

input_layer = Input(shape=(32, 32, 3))  # Updated input shape
x = Conv2D(64, (7, 7), padding='same', activation='relu')(input_layer)
x = MaxPooling2D((2, 2))(x)

for i in range(2):  # You can adjust the number of modules and blocks
    x = inception_module(x, [64, 128, 32])
    x = residual_block(x, 128)

x = GlobalAveragePooling2D()(x)
output_layer = Dense(10, activation='softmax')(x)  # Changed activation to softmax for classification

model = Model(inputs=input_layer, outputs=output_layer)

model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy')  # Changed loss function

datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
)

batch_size = 64
epochs = 20

history = model.fit(datagen.flow(x_train, y_train, batch_size=batch_size), epochs=epochs, validation_data=(x_test, y_test))

test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print(f"Test accuracy: {test_acc}")

Epoch 1/20
782/782 [==============================] - 1298s 2s/step - loss: 1.4710 - val_loss: 2.3250
Epoch 2/20
 44/782 [>.............................] - ETA: 20:25 - loss: 1.2263